<a href="https://colab.research.google.com/github/khyatimirani/AI-Experiments/blob/main/pcos_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai pymupdf tqdm transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
PDF_PATH

Saving pcos-lifestyle.pdf to pcos-lifestyle.pdf


'pcos-lifestyle.pdf'

In [3]:
import fitz  # PyMuPDF

def extract_pdf_text(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

full_text = extract_pdf_text(PDF_PATH)
print(len(full_text))

38847


In [4]:
def chunk_text(text, chunk_size=200, overlap=70):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(full_text)
len(chunks)

299

In [5]:
import os
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

In [6]:
PROMPT_TEMPLATE = """
You are generating a patient-facing medical Q&A dataset.

SOURCE:
The TEXT comes from a doctor-written PDF about PCOS lifestyle management, including daily habits, diet, activity, sleep, stress, and general health guidance.

TASK:
Convert the TEXT into realistic chat conversations where a woman with PCOD/PCOS asks lifestyle-related questions, and a medical assistant answers using ONLY the information from the TEXT.

IMPORTANT SOURCES TO USE:
- FAQ sections → convert directly into Q&A
- doctor recommendations → turn into patient questions
- tables or schedules → convert into practical advice answers
- lifestyle tips related to:
  * diet or eating habits
  * exercise or physical activity
  * sleep routines
  * stress management
  * daily habits
  * general health practices for PCOS

IGNORE:
- references
- acknowledgements
- author info
- purely theoretical background without actionable guidance

PATIENT CONTEXT:
The user is:
- a woman diagnosed with PCOD/PCOS
- trying to manage her condition through daily lifestyle changes
- unsure what habits are helpful or harmful
- looking for simple, practical guidance
- may ask about food, exercise, sleep, stress, routine, or general health choices

QUESTION RULES:
- Natural, patient-style wording
- Use first-person where appropriate
- Focus on lifestyle, habits, routine, diet, activity, sleep, or stress
- Avoid medical jargon
- Generate multiple diverse questions if TEXT supports it

ANSWER RULES:
- Use ONLY the TEXT
- Do NOT add outside medical knowledge
- Keep answers practical and easy to follow
- Keep answers SHORT (2–4 sentences max)
- Include numbers, limits, or durations only if mentioned in TEXT
- Mention uncertainty if advice is general
- Do NOT invent recommendations

OUTPUT FORMAT:
Return ONLY valid JSON array.

Each item must be:

{
  "messages":[
    {"role":"user","content":"question"},
    {"role":"assistant","content":"answer"}
  ]
}

If no usable lifestyle guidance exists in TEXT, return [].

TEXT:
"""

In [7]:
import json
import time

def extract_evidence(chunk):

    prompt = PROMPT_TEMPLATE + chunk

    response = client.chat.completions.create(
        model="gpt-4.1-mini",   # or your preferred model
        messages=[{"role":"user","content":prompt}],
        temperature=0
    )

    content = response.choices[0].message.content.strip()

    try:
        data = json.loads(content)
        if isinstance(data, dict):
            return []
        if isinstance(data, list):
            return data
    except:
        return []

    return []

In [8]:
import time
import json

def generate_qa_with_retry(chunk, retries=3):

    for attempt in range(retries):
        try:
            raw_output = generate_qa(chunk)
            # Attempt to parse the string output as JSON
            response_obj = json.loads(raw_output)
            qa_pairs = response_obj.get("qa_data", [])
            if not isinstance(qa_pairs, list):
                print(f"Warning: 'qa_data' is not a list. Received: {qa_pairs}")
                qa_pairs = [] # Ensure it's a list even if model deviates
            return qa_pairs
        except json.JSONDecodeError as e:
            print(f"Retry {attempt+1} due to JSON decoding error: {e}. Raw output: {raw_output}") # Print full raw_output
        except Exception as e:
            print(f"Retry {attempt+1} due to error: {e}. Raw output: {raw_output}")
        time.sleep(5) # Wait before retrying

    return None

In [9]:
from tqdm.auto import tqdm
import json, time

OUTPUT_FILE = "pcod_diet_qa_dataset.jsonl"

all_records = 0
seen_questions = set()   # prevent duplicates

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    for chunk in tqdm(chunks):

        records = extract_evidence(chunk)   # keep same function if prompt updated

        # Ensure list
        if not isinstance(records, list):
            continue

        for r in records:

            # Validate chat structure
            if (
                isinstance(r, dict)
                and "messages" in r
                and isinstance(r["messages"], list)
                and len(r["messages"]) == 2
            ):
                user_msg, assistant_msg = r["messages"]

                if (
                    user_msg.get("role") == "user"
                    and assistant_msg.get("role") == "assistant"
                ):

                    question = user_msg.get("content", "").strip()
                    answer = assistant_msg.get("content", "").strip()

                    # Basic quality filters
                    if len(question) < 8 or len(answer) < 15:
                        continue
                    if "?" not in question:
                        continue

                    # Deduplicate
                    key = question.lower()
                    if key in seen_questions:
                        continue
                    seen_questions.add(key)

                    f.write(json.dumps(r, ensure_ascii=False) + "\n")
                    all_records += 1

        f.flush()
        time.sleep(0.8)   # avoid rate limits

print("Total records:", all_records)

  0%|          | 0/299 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
import json
import pandas as pd

# Replace with your filename
file_name = "qa_dataset.jsonl"

rows = []

with open(file_name, "r") as f:
    for line in f:
        data = json.loads(line)
        messages = data["messages"]

        # Extract user and assistant messages
        user_msg = next((m["content"] for m in messages if m["role"] == "user"), None)
        assistant_msg = next((m["content"] for m in messages if m["role"] == "assistant"), None)

        rows.append({
            "user_message": user_msg,
            "assistant_response": assistant_msg
        })

# Convert to DataFrame
df = pd.DataFrame(rows)

df

FileNotFoundError: [Errno 2] No such file or directory: 'qa_dataset.jsonl'

In [ ]:
files.download("pcos_evidence_dataset.jsonl")

# Task
Replace the OpenAI API call in the `extract_evidence` function with a local Hugging Face model, specifically `Mistral-7B-Instruct-v0.2`, configured for 4-bit quantization, to generate Q&A pairs from the PDF text chunks, and then review the resulting Q&A dataset.

## Load Hugging Face Model

### Subtask:
Load a suitable Hugging Face model (e.g., Mistral-7B-Instruct-v0.2) and its tokenizer, configured for 4-bit quantization using `bitsandbytes` to optimize memory usage.


**Reasoning**:
First, I'll import all the necessary libraries and modules for loading a Hugging Face model with 4-bit quantization.



In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from bitsandbytes.quantization import BitsAndBytesConfig
import torch
import os

print("Libraries imported successfully.")

ModuleNotFoundError: No module named 'bitsandbytes.quantization'

**Reasoning**:
The previous error indicated that 'BitsAndBytesConfig' could not be found under `bitsandbytes.quantization`. This class is typically imported directly from the `transformers` library after `bitsandbytes` has been installed. I will correct the import statement to reflect this.



In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import os

print("Libraries imported successfully.")

Libraries imported successfully.


**Reasoning**:
Now that the libraries are correctly imported, I will define the model ID and create the `BitsAndBytesConfig` object as specified in the subtask instructions to prepare for loading the quantized model.



In [12]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

print("Model ID and BitsAndBytesConfig initialized.")

Model ID and BitsAndBytesConfig initialized.


**Reasoning**:
With the `BitsAndBytesConfig` initialized, I will now load the tokenizer and the 4-bit quantized model using `AutoTokenizer.from_pretrained` and `AutoModelForCausalLM.from_pretrained` respectively, as per the subtask instructions. I will use `userdata.get("HF_TOKEN")` for authentication.



In [13]:
HF_TOKEN = userdata.get("HF_TOKEN")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)

print("Tokenizer and Model loaded successfully.")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

KeyboardInterrupt: 